# Whakoom collection — exploration

Starter notebook over the views built by `uv run wk analyze` (Phase 7).

- Views live in `data/whakoom.duckdb`; the facts stay in SQLite (`data/whakoom.db`).
- The views reference the SQLite database as catalog `wh`, so every session must
  re-`ATTACH` it first (see next cell).
- Regenerate everything after a scrape with `uv run wk run-all`.

It answers the three Phase 7 questions on real data:

1. Top publishers by year
2. Score distribution
3. Score / ownership trend of one series over ≥ 2 runs

In [1]:
from pathlib import Path

import duckdb

# Paths are relative to this notebook (analysis/).
DUCKDB_FILE = Path("../data/whakoom.duckdb")
SQLITE_FILE = Path("../data/whakoom.db").resolve()

con = duckdb.connect(str(DUCKDB_FILE), read_only=True)
con.execute(f"ATTACH '{SQLITE_FILE.as_posix()}' AS wh (TYPE sqlite, READ_ONLY)")


def query(sql: str):
    """Runs a query and returns a pandas DataFrame."""
    return con.execute(sql).df()


print(con.execute("SELECT COUNT(*) AS series FROM wh.series").fetchone())

(1182,)


## (a) Top publishers by year

Distinct titles licensed per publisher, from the year license lists
(`v_publisher_share_by_year`).

In [2]:
query("""
    SELECT
        year,
        publisher,
        SUM(titles) AS titles,
        ROUND(100.0 * SUM(titles) / SUM(SUM(titles)) OVER (PARTITION BY year), 1) AS share_pct
    FROM v_publisher_share_by_year
    WHERE year >= 2023
    GROUP BY year, publisher
    QUALIFY ROW_NUMBER() OVER (PARTITION BY year ORDER BY SUM(titles) DESC) <= 5
    ORDER BY year DESC, titles DESC
""")

,year,publisher,titles,share_pct
0,2026,Moztros,10.0,20.4
1,2026,Ivrea,7.0,14.3
2,2026,Devir Iberia,7.0,14.3
3,2026,Panini Comics España,7.0,14.3
4,2026,Distrito Manga,3.0,6.1
5,2025,Planeta Cómic,46.0,17.8
6,2025,Ivrea,34.0,13.1
7,2025,Norma Editorial,34.0,13.1
8,2025,Milky Way Ediciones,30.0,11.6
9,2025,Arechi Manga,19.0,7.3


## (b) Score distribution

How the 1,183 scraped series rate on Whakoom (community rating, 0–5).

In [3]:
query("""
    SELECT
        CASE
            WHEN rating IS NULL THEN 'unrated'
            WHEN rating >= 4.5 THEN '4.5-5.0'
            WHEN rating >= 4.0 THEN '4.0-4.4'
            WHEN rating >= 3.5 THEN '3.5-3.9'
            WHEN rating >= 3.0 THEN '3.0-3.4'
            ELSE 'below 3.0'
        END AS bucket,
        COUNT(*) AS titles,
        ROUND(AVG(rating_count), 1) AS avg_votes
    FROM wh.series
    GROUP BY 1
    ORDER BY 1
""")

,bucket,titles,avg_votes
0,3.0-3.4,62,9.0
1,3.5-3.9,101,15.1
2,4.0-4.4,182,23.3
3,4.5-5.0,330,23.2
4,below 3.0,124,0.7
5,unrated,383,NaN


## (c) Score / ownership trend of one series

`v_score_history` joins `series_observations` with `scrape_runs`: one row per
(series, run). Set `SERIES` to any name and re-run. Values only move between
scrape runs — run `uv run wk series --force` to add an observation batch.

In [4]:
SERIES = "Rosen Blood"

query(f"""
    SELECT
        observed_at,
        rating,
        rating_count,
        ownership_count,
        volumes_count
    FROM v_score_history
    WHERE series_name = '{SERIES}'
    ORDER BY observed_at
""")

,observed_at,rating,rating_count,ownership_count,volumes_count
0,2026-08-16T21:00:10Z,4.3,16,198,5


Series with the most rating votes right now, for picking another trend target:

In [5]:
query("""
    SELECT name, rating, rating_count, ownership_count
    FROM wh.series
    ORDER BY rating_count DESC
    LIMIT 10
""")

,name,rating,rating_count,ownership_count
0,"¡¡No te rindas, Nakamura!!",4.5,370,2
1,Orange,4.6,278,3
2,"Yona, Princesa del Amanecer",4.8,268,3
3,All You Need Is Kill,4.3,208,1
4,Aoha Ride,4.7,201,2
5,Confuso primer amor,4.9,152,2
6,Réquiem por el Rey de la Rosa,4.8,147,1
7,"Nagahama To Be, or Not To Be",4.7,129,995
8,Nana,4.8,119,2
9,Marmalade Boy. Edición especial,4.5,113,1


## Further exploration

- `v_titles_by_magazine` — titles serialized in each magazine (canonical name).
- `v_rating_by_publisher` — average rating / votes / ownership per publisher.
- `v_list_overlap` — series appearing in more than one list.